# 05 — Colab RL Training

**Columbia MSDS STATGR5293 — Optimized LLM Planning with Memory**

Runs the full PPO training pipeline on Google Colab for **any compressor**.
Set `COMPRESSOR` and `TRAINING_CONFIG` in the **Config** cell below, then run all cells top-to-bottom.

| Section | What it does |
|---|---|
| 1. Config | Choose compressor, training preset, Drive paths |
| 2. Setup | GPU check, clone repos, install, Drive mount, API keys |
| 3. Requests | Generate synthetic `UserRequest` files |
| 4. Smoke test | One episode before committing to full training |
| 5. TensorBoard | Launch live reward monitor |
| 6. Training | PPO training loop |
| 7. Diagnostics | Episode metrics, PPO update metrics, trajectory preview, MCTS stats |
| 8. Checkpoints | List, verify, manage |
| 9. Evaluation | Deterministic + LLM-judge eval |
| 10. Bundle | Package and share the run |

**Runtime:** `Runtime → Change runtime type → T4 GPU`.  
Identity/LLM compressors: ~20–40 min for 50 k steps.  
Transformer/MCTS-GAT: ~60–120 min (trains neural weights on GPU).

---
## 1  Config — edit this cell before running anything else

In [ ]:
# ── Which compressor to train ─────────────────────────────────────────────────
#
#  COMPRESSOR          | TRAINING_CONFIG | Agent mode        | Notes
#  --------------------|-----------------|-------------------|-------------------
#  identity            | ppo_colab       | compressor        | Fastest. No GPU. PPO trains value head only.
#  llm_prompt          | ppo_colab       | compressor        | LLM summarisation. No neural weights. Needs API key.
#  transformer         | ppo_colab       | compressor        | Flan-T5 fine-tuned on GPU. ~2× identity runtime.
#  structured_selective| ppo_colab       | compressor        | Section-aware cross-attention (LoRA). GPU recommended.
#  hybrid              | ppo_colab       | compressor        | LLM summary + structured extraction combined.
#  llm_mcts            | ppo_mcts        | mcts_compressor   | LLM compressor + MCTS tree search. Needs API key.
#  mcts_gat            | ppo_mcts        | mcts_compressor   | Graph-attention distiller over MCTS tree. GPU required.
#
COMPRESSOR      = 'identity'   # <── change this
TRAINING_CONFIG = 'ppo_colab'  # ppo_colab | ppo_mcts (for llm_mcts / mcts_gat) | ppo_debug (8-step local test)

import datetime as _dt
RUN_NAME = f'{COMPRESSOR}_{_dt.datetime.now().strftime("%Y%m%d_%H%M%S")}'  # appears in manifest + Streamlit selector

# ── Paths — adjust for your Drive layout ─────────────────────────────────────
# Option A: repos cloned to /content/ (default)
REPO_DIR         = '/content/optimized-llm-planning-memory'
TRAVEL_WORLD_DIR = '/content/my-travel-world'

# Option B: repos on Google Drive (uncomment and set your path)
# DRIVE_ROOT       = '/content/drive/MyDrive/STATGR5293 - GenAI Final Project'
# REPO_DIR         = f'{DRIVE_ROOT}/optimized-llm-planning-memory'
# TRAVEL_WORLD_DIR = f'{DRIVE_ROOT}/my-travel-world'

# ── Derived (do not edit) ─────────────────────────────────────────────────────
import sys, os
from pathlib import Path

OUTPUT_DIR   = Path(REPO_DIR) / 'outputs'
TRAINING_DIR = OUTPUT_DIR / 'training'
EPISODES_DIR = OUTPUT_DIR / 'episodes'
sys.path.insert(0, str(Path(REPO_DIR) / 'src'))

IS_MCTS = COMPRESSOR in ('llm_mcts', 'mcts_gat')

print(f'Compressor     : {COMPRESSOR}')
print(f'Training config: {TRAINING_CONFIG}')
print(f'Run name       : {RUN_NAME}')
print(f'MCTS mode      : {IS_MCTS}')
print(f'Repo dir       : {REPO_DIR}')

---
## 2  Setup

In [ ]:
# Verify GPU availability
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')
    print('Go to Runtime → Change runtime type → GPU → T4')

In [ ]:
# ── Clone / update repos ──────────────────────────────────────────────────────
# Skip this section if your repos already live on Google Drive (Option B in Config).

REPO_URL         = 'https://github.com/YOUR_ORG/optimized-llm-planning-memory.git'
TRAVEL_WORLD_URL = 'https://github.com/YOUR_ORG/my-travel-world.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Main repo present — pulling...')
    !git -C {REPO_DIR} pull

if not os.path.exists(TRAVEL_WORLD_DIR):
    !git clone {TRAVEL_WORLD_URL} {TRAVEL_WORLD_DIR}
else:
    print('Travel world present — pulling...')
    !git -C {TRAVEL_WORLD_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

In [ ]:
# ── Install order matters ─────────────────────────────────────────────────────
# pip does not read [tool.uv.sources], so the travel simulator must be installed
# BEFORE the main package — otherwise pip looks for 'travel-world' on PyPI and
# fails with "Could not find a version that satisfies the requirement travel-world".

# If repos are on Google Drive instead of /content/, set this variable:
# TRAVEL_WORLD_DIR = '/content/drive/MyDrive/STATGR5293 - GenAI Final Project/my-travel-world'
# REPO_DIR         = '/content/drive/MyDrive/STATGR5293 - GenAI Final Project/optimized-llm-planning-memory'

# 1. Simulator first — main package dependency
!pip install -q -e {TRAVEL_WORLD_DIR}

# 2. Main package (travel-world already satisfies its own dependency now)
!pip install -q -e {REPO_DIR}[dev]

!pip install -q tensorboard
print('Installation complete.')

### API Keys

Store keys via `Tools → Secrets` (🔑). Required: `OPENAI_API_KEY`. Optional: `ANTHROPIC_API_KEY` (only if switching the agent to a Claude model).

In [ ]:
from google.colab import userdata

def _load_secret(name: str, required: bool = False) -> str | None:
    try:
        val = userdata.get(name)
        os.environ[name] = val
        print(f'  {name}: loaded ({len(val)} chars)')
        return val
    except Exception:
        msg = 'MISSING — add it via the 🔑 Secrets panel'
        if required:
            msg += '  ← REQUIRED'
        print(f'  {name}: {msg}')
        return None

print('Loading API keys from Colab Secrets...')
_load_secret('OPENAI_API_KEY',     required=True)   # agent + LLM compressors
_load_secret('ANTHROPIC_API_KEY',  required=False)  # optional: swap agent to Claude

# LLM used by the ReAct planning agent
os.environ.setdefault('AGENT_LLM_MODEL_ID', 'openai/gpt-4o-mini')
print('Agent LLM:', os.environ['AGENT_LLM_MODEL_ID'])

---
## 3  Google Drive Mount (for checkpoint persistence)

Colab sessions reset after ~12 h of inactivity. Mount Drive so checkpoints survive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Persist all outputs to Drive so they survive session resets
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/optllm_training'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

local_out = str(OUTPUT_DIR)
if not os.path.exists(local_out):
    os.symlink(DRIVE_OUTPUT_DIR, local_out)
    print(f'Symlinked outputs/ → {DRIVE_OUTPUT_DIR}')
else:
    print('outputs/ already exists')

---
## 3  Generate User Requests

In [ ]:
# Preview the resolved Hydra config without launching training
!python {REPO_DIR}/scripts/run_training.py \
    compressor={COMPRESSOR} \
    training={TRAINING_CONFIG} \
    --cfg job

### Generate requests (skip if already generated)

In [ ]:
!python scripts/generate_user_requests.py \
    n_train=40 n_val=10 n_test=10 \
    project.seed=42

# Verify
from pathlib import Path
for split in ('train', 'val', 'test'):
    n = len(list(Path(f'data/user_requests/{split}').glob('*.json')))
    print(f'  {split}: {n} requests')

---
## 4  Smoke Test — Single Episode

In [ ]:
!python {REPO_DIR}/scripts/run_episode.py \
    compressor={COMPRESSOR} \
    agent.max_steps=5 \
    project.seed=42

In [ ]:
import json
from pathlib import Path

episode_files = sorted(Path('outputs/episodes').glob('*.json'))
if not episode_files:
    print('No episode logs found — check the error output above.')
else:
    log = json.loads(episode_files[-1].read_text())
    print(f"episode_id : {log['episode_id']}")
    print(f"success    : {log['success']}")
    print(f"total_steps: {log['total_steps']}")
    print(f"error      : {log.get('error')}")
    rc = log.get('reward_components', {})
    print(f"total_reward          : {rc.get('total_reward', 0):.3f}")
    print(f"hard_constraint_score : {rc.get('hard_constraint_score', 0):.3f}")
    print(f"tool_efficiency_score : {rc.get('tool_efficiency_score', 0):.3f}")

---
## 5  TensorBoard

Launch **before** training so it updates live.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/logs

---
## 6  PPO Training

Interrupt with **Runtime → Interrupt execution** — the last checkpoint is preserved on Drive.

In [ ]:
!python {REPO_DIR}/scripts/run_training.py \
    compressor={COMPRESSOR} \
    training={TRAINING_CONFIG} \
    training.run_name={RUN_NAME} \
    project.seed=42

### Resuming from a checkpoint

If the session restarted, re-run cells 1–5, then run the cell below to resume PPO training from the last saved checkpoint.

In [ ]:
ckpt_dir = OUTPUT_DIR / 'checkpoints'
zips = sorted(ckpt_dir.glob('ppo_compressor_*_steps.zip'))
if not zips:
    print('No checkpoints found — run training first.')
else:
    latest = zips[-1]
    print(f'Resuming from: {latest}')
    !python {REPO_DIR}/scripts/run_training.py \
        compressor={COMPRESSOR} \
        training={TRAINING_CONFIG} \
        training.run_name={RUN_NAME} \
        training.resume_from={latest} \
        project.seed=42

---
## 7  Diagnostics

### 7a  Episode metrics (from JSONL logger)

In [ ]:
import pandas as pd
from optimized_llm_planning_memory.training.run_manifest import list_manifests
from optimized_llm_planning_memory.training.run_logger import load_episode_metrics, load_ppo_metrics

# Resolve the most recent training run
manifests = list_manifests(TRAINING_DIR)
if not manifests:
    print('No training runs found. Complete training first.')
    RUN_ID = None
    ep_df = pd.DataFrame()
else:
    m = manifests[0]
    RUN_ID = m.run_id
    print(f'Run ID      : {RUN_ID}')
    print(f'Run name    : {m.run_name}')
    print(f'Compressor  : {m.compressor_type}')
    print(f'Agent mode  : {m.agent_mode}')
    print(f'Timesteps   : {m.num_timesteps:,}')
    print(f'Created at  : {m.created_at}')

    ep_records = load_episode_metrics(RUN_ID, TRAINING_DIR)
    if not ep_records:
        print('\nNo episode records yet.')
        ep_df = pd.DataFrame()
    else:
        ep_df = pd.DataFrame([r.model_dump() for r in ep_records])
        pd.set_option('display.max_columns', None)
        pd.set_option('display.float_format', '{:.3f}'.format)
        print(f'\n{len(ep_df)} episode records')
        display(ep_df.tail(10))

In [ ]:
import matplotlib.pyplot as plt

if ep_df.empty:
    print('No episode data to plot.')
else:
    window = max(1, len(ep_df) // 20)

    reward_cols = [
        ('total_reward',              'Total Reward'),
        ('hard_constraint_score',     'Hard Constraint Score'),
        ('soft_constraint_score',     'Soft Constraint Score'),
        ('tool_efficiency_score',     'Tool Efficiency'),
        ('tool_failure_penalty',      'Tool Failure Penalty'),
        ('logical_consistency_score', 'Logical Consistency'),
    ]
    # Add optional columns if present
    if 'terminal_itinerary_score' in ep_df.columns:
        reward_cols.append(('terminal_itinerary_score', 'Terminal Itinerary Score'))

    present = [(c, t) for c, t in reward_cols if c in ep_df.columns]
    ncols = 3
    nrows = (len(present) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes.flatten()

    for ax, (col, title) in zip(axes, present):
        ax.plot(ep_df[col].values, alpha=0.25, color='steelblue', linewidth=0.8)
        ax.plot(ep_df[col].rolling(window, min_periods=1).mean().values,
                color='steelblue', linewidth=2, label=f'MA-{window}')
        ax.set_title(title)
        ax.set_xlabel('Episode')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    for ax in axes[len(present):]:
        ax.set_visible(False)

    plt.suptitle(f'Episode Metrics — {COMPRESSOR} / {RUN_NAME}', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'training_diagnostics.png'), dpi=150)
    plt.show()
    print('Saved → outputs/training_diagnostics.png')

In [ ]:
if not ep_df.empty:
    last_n = min(50, len(ep_df))
    recent = ep_df.tail(last_n)
    print(f'=== Last {last_n} episodes ===')
    for col, label in [
        ('success',                  'Success rate'),
        ('total_reward',             'Avg total reward'),
        ('hard_constraint_score',    'Avg hard constraint'),
        ('soft_constraint_score',    'Avg soft constraint'),
        ('tool_efficiency_score',    'Avg tool efficiency'),
        ('logical_consistency_score','Avg logical consistency'),
        ('total_steps',              'Avg steps / episode'),
        ('tool_calls_total',         'Avg tool calls'),
        ('num_compressions',         'Avg compressions'),
    ]:
        if col in recent.columns:
            val = recent[col].mean()
            if col == 'success':
                print(f'  {label:<28}: {val:.1%}')
            else:
                print(f'  {label:<28}: {val:.3f}')

### 7b  PPO update diagnostics

| Metric | Healthy range | Red flag |
|---|---|---|
| `approx_kl` | < 0.02 | > 0.05 — LR too high |
| `clip_fraction` | 0.1–0.2 | > 0.3 |
| `explained_variance` | trending → 1.0 | stuck near 0 |
| `policy_loss` | decreasing | exploding / nan |

In [ ]:
if RUN_ID:
    ppo_records = load_ppo_metrics(RUN_ID, TRAINING_DIR)
    if not ppo_records:
        print('No PPO update records yet (need enough timesteps for at least one update).')
    else:
        ppo_df = pd.DataFrame([r.model_dump() for r in ppo_records])
        print(f'{len(ppo_df)} PPO update records')
        display(ppo_df.round(5))

        if len(ppo_df) > 1:
            ppo_plot_cols = [c for c in
                ['policy_loss', 'value_loss', 'approx_kl', 'clip_fraction', 'explained_variance', 'entropy_loss']
                if c in ppo_df.columns]
            fig, axes = plt.subplots(1, len(ppo_plot_cols), figsize=(4 * len(ppo_plot_cols), 3))
            if len(ppo_plot_cols) == 1:
                axes = [axes]
            for ax, col in zip(axes, ppo_plot_cols):
                ax.plot(ppo_df[col].values, marker='o', markersize=3)
                ax.set_title(col)
                ax.set_xlabel('PPO update')
                ax.grid(alpha=0.3)
            plt.suptitle('PPO Update Metrics', fontsize=12)
            plt.tight_layout()
            plt.show()

### 7c  Episode trajectory preview

Inspect the most recent saved episode: ReAct steps, tool calls, compressed states.

In [ ]:
import json

ep_files = sorted(EPISODES_DIR.glob('ep_*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
print(f'{len(ep_files)} episode JSON files')

if ep_files:
    raw = json.loads(ep_files[0].read_text(encoding='utf-8'))

    print(f'\nepisode_id        : {raw["episode_id"]}')
    print(f'request_id        : {raw["request_id"]}')
    print(f'agent_mode        : {raw["agent_mode"]}')
    print(f'total_steps       : {raw["total_steps"]}')
    print(f'success           : {raw["success"]}')
    print(f'termination_reason: {raw.get("termination_reason", "N/A")}')

    print('\nReward components:')
    for k, v in raw.get('reward_components', {}).items():
        if isinstance(v, (int, float)):
            print(f'  {k:<35} {v:.4f}')

    stats = raw.get('tool_call_stats', {})
    print(f'\nTool calls: total={stats.get("total_calls",0)}, '
          f'success={stats.get("successful_calls",0)}, '
          f'failed={stats.get("failed_calls",0)}')

    print(f'\n── Trajectory ({len(raw.get("trajectory",{}).get("steps",[]))} steps) ──')
    for step in raw.get('trajectory', {}).get('steps', []):
        print(f'\nStep {step["step_index"]}')
        print(f'  Thought : {step["thought"][:120]}')
        if step.get('action'):
            act = step['action']
            print(f'  Action  : {act.get("tool_name","?")}({str(act.get("arguments",{}))[:80]})')
        if step.get('observation'):
            obs = step['observation']
            ok = "OK" if obs.get("success") else "FAIL"
            print(f'  Obs [{ok}]: {str(obs.get("result") or obs.get("error_message",""))[:120]}')

    compressed = raw.get('compressed_states', [])
    if compressed:
        print(f'\n── Compressed states ({len(compressed)}) ──')
        for cs in compressed:
            print(f'\nStep {cs["step_index"]} (method: {cs.get("compression_method","?")})')
            print(f'  Decisions       : {cs.get("decisions_made", [])}')
            print(f'  Open questions  : {cs.get("open_questions", [])}')
            sketch = cs.get("current_itinerary_sketch", "")
            print(f'  Itinerary sketch: {sketch[:200]}')

### 7d  MCTS search diagnostics (llm_mcts / mcts_gat only)

In [ ]:
if not IS_MCTS:
    print(f'MCTS diagnostics not applicable for compressor="{COMPRESSOR}". '
          f'Set COMPRESSOR to "llm_mcts" or "mcts_gat" to see MCTS stats.')
elif not ep_files:
    print('No episode files to inspect.')
else:
    # Collect mcts_stats from all saved episode JSONs
    mcts_rows = []
    for f in ep_files:
        try:
            r = json.loads(f.read_text(encoding='utf-8'))
            ms = r.get('mcts_stats')
            if ms:
                mcts_rows.append(ms)
        except Exception:
            pass

    if not mcts_rows:
        print('No mcts_stats found in episode files. '
              'Ensure episode_save_freq > 0 was set during training.')
    else:
        mcts_df = pd.DataFrame(mcts_rows)
        print(f'{len(mcts_df)} episodes with MCTS stats\n')
        display(mcts_df.describe().round(3))

        fig, axes = plt.subplots(1, 3, figsize=(13, 3))
        for ax, col, title in [
            (axes[0], 'nodes_explored',      'Nodes Explored'),
            (axes[1], 'root_value',          'Root Q-value'),
            (axes[2], 'avg_branching_factor','Avg Branching Factor'),
        ]:
            if col in mcts_df.columns:
                ax.plot(mcts_df[col].values, marker='o', markersize=3)
                ax.set_title(title)
                ax.set_xlabel('Episode')
                ax.grid(alpha=0.3)
        plt.suptitle('MCTS Search Statistics', fontsize=12)
        plt.tight_layout()
        plt.show()

### 7e  Reward predictor diagnostics

In [ ]:
import torch
from pathlib import Path

# The RewardPredictorCallback saves weights alongside the compressor checkpoint
rp_files = sorted(Path('outputs/checkpoints').rglob('reward_predictor.pt'))
if not rp_files:
    print('No reward predictor checkpoint yet (fit() requires ≥ 20 episodes).')
else:
    ckpt = torch.load(rp_files[-1], map_location='cpu')
    weight = ckpt['model_state']['weight'].squeeze()   # shape (5,)
    bias   = ckpt['model_state']['bias'].item()
    feature_names = ckpt['feature_names']
    print('RewardPredictorComponent weights (latest checkpoint):')
    for name, w in zip(feature_names, weight.tolist()):
        print(f'  {name:<30s}  {w:+.4f}')
    print(f'  bias                            {bias:+.4f}')
    print(f'  trained on {ckpt["n_episodes_trained"]} episodes')

---
## 8  Checkpoints

In [ ]:
from pathlib import Path

ckpt_dir = Path('outputs/checkpoints')
zips = sorted(ckpt_dir.glob('ppo_compressor_*_steps.zip'))
print(f'SB3 checkpoints ({len(zips)} total):')
for z in zips:
    size_mb = z.stat().st_size / 1e6
    print(f'  {z.name:<50s}  {size_mb:.1f} MB')

final = ckpt_dir / 'final'
if final.exists():
    print(f'\nFinal checkpoint: {final}')
    for f in sorted(final.rglob('*')):
        if f.is_file():
            print(f'  {f.relative_to(final)}')

In [ ]:
# Verify the SB3 PPO model loads cleanly
from stable_baselines3 import PPO

ckpt_path = OUTPUT_DIR / 'checkpoints' / 'final' / 'ppo_model.zip'
if ckpt_path.exists():
    try:
        model = PPO.load(str(ckpt_path), device='cpu')
        print(f'PPO checkpoint loaded OK')
        print(f'  Policy           : {type(model.policy).__name__}')
        print(f'  Observation space: {model.observation_space}')
        print(f'  Action space     : {model.action_space}')
    except Exception as e:
        print(f'Load failed: {e}')
else:
    print(f'No final checkpoint at {ckpt_path}. Run training first.')

---
## 9  Evaluation

Uses `+run_id` to auto-resolve the checkpoint from `manifest.json` — no manual path needed.

In [ ]:
if RUN_ID:
    # Deterministic metrics only (no LLM judge — fast, no API key needed)
    !python {REPO_DIR}/scripts/run_evaluation.py \
        +run_id={RUN_ID} \
        eval.deterministic_only=true

    # Full evaluation with LLM judge (uncomment when ready):
    # !python {REPO_DIR}/scripts/run_evaluation.py +run_id={RUN_ID}
else:
    print('No RUN_ID — run training first (Section 6).')

In [ ]:
from pathlib import Path
import json

eval_dir = OUTPUT_DIR / 'eval_results'
eval_files = sorted(eval_dir.glob('*.json')) if eval_dir.exists() else []
if not eval_files:
    print('No eval results yet.')
else:
    results = json.loads(eval_files[-1].read_text())
    print(json.dumps(results, indent=2))

---
## Appendix — Troubleshooting

| Problem | Fix |
|---|---|
| `Could not find a version that satisfies the requirement travel-world` | Install order is wrong — re-run the pip cell. The simulator (`TRAVEL_WORLD_DIR`) must be installed **before** `.[dev]`. |
| `ModuleNotFoundError: No module named 'hydra'` | Same root cause as above — the `.[dev]` install failed partway through. Fix install order and re-run. |
| `ModuleNotFoundError: travel_world` | Check `TRAVEL_WORLD_DIR` points to the cloned simulator repo; re-run the pip cell. |
| `ANTHROPIC_API_KEY not found` | Add it via the 🔑 Secrets panel; re-run the secrets cell. |
| TensorBoard shows no data | Training must write at least one episode; check the training cell output for errors. |
| CUDA out of memory | Reduce `training.ppo.batch_size` to 16 or set `training.n_envs=1`. |
| Session disconnected mid-run | Outputs are mirrored to Drive; re-run from the **Resuming from a checkpoint** cell. |
| Episode always fails with max_steps | Increase `agent.max_steps` or check that `AGENT_LLM_MODEL_ID` has a valid API key. |

---
## 10  Bundle & Share This Run

Packages `manifest.json`, JSONL logs, and the final checkpoint into a single `.tar.gz`.

In [ ]:
import sys
sys.path.insert(0, "src")

from pathlib import Path
from optimized_llm_planning_memory.training.run_manifest import list_manifests
from optimized_llm_planning_memory.utils.colab_utils import (
    bundle_run, upload_to_drive, download_bundle, estimate_run_size
)

# Find the latest training run
manifests = list_manifests("outputs/training")
if not manifests:
    print("No training runs found. Complete training first.")
else:
    latest = manifests[0]  # newest first
    run_id = latest.run_id
    print(f"Latest run : {run_id}")
    print(f"Compressor : {latest.compressor_type}")
    print(f"Run name   : {latest.run_name}")
    print(f"Timesteps  : {latest.num_timesteps:,}")
    print()

    # Estimate storage before bundling
    sizes = estimate_run_size(run_id)
    for k, v in sizes.items():
        print(f"  {k:<25} {v:.1f} MB")


In [ ]:
# Create the bundle archive
# The archive includes: manifest.json, ppo_metrics.jsonl,
# episode_metrics.jsonl, final checkpoint, and TensorBoard logs (if < 50 MB).

bundle_path = bundle_run(
    run_id=run_id,
    output_dir="outputs",
    bundle_dir="outputs/bundles",
)
print(f"Bundle: {bundle_path}")


In [ ]:
# Upload bundle to Google Drive (optional)
# Other team members can download this and unpack it locally for comparison.

DRIVE_BUNDLES_DIR = "/content/drive/MyDrive/optllm_training/bundles"

upload_to_drive(bundle_path, drive_dir=DRIVE_BUNDLES_DIR)

# Trigger in-browser download (works in Colab)
download_bundle(run_id, bundle_dir="outputs/bundles")


### Running evaluation directly from a run_id

Instead of manually specifying a checkpoint path, use  to let the
evaluation script resolve the checkpoint automatically from :



The script reads  to find the
checkpoint and infer the compressor type — no manual path needed.
